# SciPy, statsmodels and the Scientific Stack

Notebook for **Chapter 18** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch18.html

Tip: change a number, predict the output, then run the cell again.

## 18.1 What SciPy and statsmodels Are, and the Mental Model

In [ ]:
import numpy as np
import scipy
import pandas as pd
import matplotlib
from scipy import stats, optimize, integrate, interpolate, linalg, signal, spatial, fft
from scipy import sparse
from scipy.cluster import vq, hierarchy

print("numpy ", np.__version__.split(".")[0] + ".x")
print("scipy ", scipy.__version__)
print("pandas", pd.__version__.split(".")[0] + ".x")
print("matplotlib", matplotlib.__version__)
print(stats.norm.cdf(0))

numpy  2.x
scipy  1.18.1
pandas 3.x
matplotlib 3.11.2
0.5


## 18.2 scipy.stats I: Distributions

A **probability distribution** is a recipe that says how likely each value is. SciPy gives you about a hundred ready-made ones. Every continuous distribution is an object with the same methods, so once you learn `norm` you know `lognorm`, `gamma`, `expon` and the rest. A *discrete* distribution (`binom`, `poisson`) uses `pmf` (probability of exactly this value) instead of `pdf` (density).

In [ ]:
import numpy as np
from scipy import stats

scores = stats.norm(loc=70, scale=10)
print("pdf at 70      :", round(scores.pdf(70), 4))
print("P(score <= 85) :", round(scores.cdf(85), 4))
print("P(score > 90)  :", round(scores.sf(90), 4))
print("90th percentile:", round(scores.ppf(0.90), 2))
rng = np.random.default_rng(0)
print("5 draws        :", scores.rvs(5, random_state=rng).round(1))

pdf at 70      : 0.0399
P(score <= 85) : 0.9332
P(score > 90)  : 0.0228
90th percentile: 82.82
5 draws        : [71.3 68.7 76.4 71.  64.6]


In [ ]:
from scipy import stats

calls = stats.poisson(mu=4)
print("P(exactly 7)  :", round(calls.pmf(7), 4))
print("P(10 or more) :", round(calls.sf(9), 5))
print("99% covered by:", int(calls.ppf(0.99)), "calls")
print("mean, std     :", calls.mean(), round(calls.std(), 3))

P(exactly 7)  : 0.0595
P(10 or more) : 0.00813
99% covered by: 9 calls
mean, std     : 4.0 2.0


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(7)
data = rng.lognormal(mean=1.2, sigma=0.5, size=800)

s, loc, scale = stats.lognorm.fit(data, floc=0)
mu, sd = stats.norm.fit(data)
print("lognorm shape s=%.3f scale=%.3f" % (s, scale))
print("normal  mu=%.3f sd=%.3f" % (mu, sd))

ll_log = stats.lognorm.logpdf(data, s, loc, scale).sum()
ll_nor = stats.norm.logpdf(data, mu, sd).sum()
print("log-likelihood lognorm: %.1f  normal: %.1f" % (ll_log, ll_nor))
print("AIC lognorm: %.1f  normal: %.1f" % (2 * 2 - 2 * ll_log, 2 * 2 - 2 * ll_nor))

print("KS p-value lognorm: %.3f" % stats.kstest(data, "lognorm", args=(s, loc, scale)).pvalue)
print("P(time > 6h):", round(stats.lognorm.sf(6, s, loc, scale), 4), "empirical:", (data > 6).mean())

lognorm shape s=0.472 scale=3.149
normal  mu=3.519 sd=1.744
log-likelihood lognorm: -1452.4  normal: -1580.0
AIC lognorm: 2908.9  normal: 3163.9
KS p-value lognorm: 0.589
P(time > 6h): 0.086 empirical: 0.08875


## 18.3 scipy.stats II: Tests, Intervals, Bootstrap, Correlation

A **hypothesis test** asks "could this difference be just luck?". It starts from a boring assumption (the *null hypothesis*, for example "both groups have the same mean"), computes a *test statistic*, and reports a **p-value**: the probability of seeing a result at least this extreme if the boring assumption were true. A small p-value (below a threshold you chose in advance, often 0.05) is evidence against the null. It is *not* the probability that the null is true, and it says nothing about how large or important the effect is.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
control = rng.normal(38, 8, 120)
variant = rng.normal(35, 9, 120)

res = stats.ttest_ind(variant, control, equal_var=False)
print("t = %.3f  p = %.4f" % (res.statistic, res.pvalue))
diff = variant.mean() - control.mean()
ci = res.confidence_interval(0.95)
print("mean diff = %.2f min, 95%% CI (%.2f, %.2f)" % (diff, ci.low, ci.high))

pooled = np.sqrt((variant.var(ddof=1) + control.var(ddof=1)) / 2)
print("Cohen d = %.2f" % (diff / pooled))

t = -2.802  p = 0.0055
mean diff = -2.82 min, 95% CI (-4.80, -0.84)
Cohen d = -0.36


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(1)
a = rng.lognormal(1.0, 0.6, 80)
b = rng.lognormal(1.3, 0.6, 80)

print("shapiro a: p = %.5f" % stats.shapiro(a).pvalue)
print("shapiro log(a): p = %.3f" % stats.shapiro(np.log(a)).pvalue)
mw = stats.mannwhitneyu(a, b, alternative="two-sided")
print("Mann-Whitney U = %.0f  p = %.5f" % (mw.statistic, mw.pvalue))
auc = mw.statistic / (len(a) * len(b))
print("P(random a beats random b) = %.3f" % auc)

shapiro a: p = 0.00000
shapiro log(a): p = 0.532
Mann-Whitney U = 2292  p = 0.00196
P(random a beats random b) = 0.358


In [ ]:
import numpy as np
from scipy import stats

table = np.array([[60, 440],
                  [85, 415]])
chi2, p, dof, expected = stats.chi2_contingency(table, correction=False)
print("chi2 = %.3f  dof = %d  p = %.4f" % (chi2, dof, p))
print("expected counts:\n", expected)
print("fisher exact p = %.4f" % stats.fisher_exact(table).pvalue)
print("rates:", table[:, 0] / table.sum(axis=1))

chi2 = 5.041  dof = 1  p = 0.0247
expected counts:
 [[ 72.5 427.5]
 [ 72.5 427.5]]
fisher exact p = 0.0309
rates: [0.12 0.17]


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(3)
data = rng.lognormal(1.2, 0.5, 60)

m, se = data.mean(), stats.sem(data)
lo, hi = stats.t.interval(0.95, df=len(data) - 1, loc=m, scale=se)
print("mean %.3f, t interval (%.3f, %.3f)" % (m, lo, hi))

res = stats.bootstrap((data,), np.median, n_resamples=5000,
                      confidence_level=0.95, method="percentile", random_state=1)
ci = res.confidence_interval
print("median %.3f, bootstrap interval (%.3f, %.3f)" % (np.median(data), ci.low, ci.high))
print("bootstrap std error: %.3f" % res.standard_error)

mean 3.819, t interval (3.183, 4.455)
median 3.132, bootstrap interval (2.939, 3.851)
bootstrap std error: 0.281


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(5)
x = np.linspace(1, 10, 40)
y = x + rng.normal(0, 0.5, 40)
y[0] = 30                                # one broken sensor reading
print("pearson  r = %.3f" % stats.pearsonr(x, y).statistic)
print("spearman r = %.3f" % stats.spearmanr(x, y).statistic)

p = np.array([0.001, 0.008, 0.039, 0.041, 0.20, 0.74])
adj = stats.false_discovery_control(p)
print("adjusted:", adj.round(3))
print("significant at 5%:", adj < 0.05)

pearson  r = 0.290
spearman r = 0.843
adjusted: [0.006 0.024 0.062 0.062 0.24  0.74 ]
significant at 5%: [ True  True False False False False]


## 18.4 scipy.optimize: Minimise, Fit, Solve

**Optimisation** means "find the inputs that make some number as small as possible". Training a model, fitting a curve, planning a route and pricing a product are all optimisation. `scipy.optimize` offers one function per family of problems. Pick by asking: is it a smooth function of a few variables (`minimize`), a model to fit to points (`curve_fit`), an equation to solve (`root`, `brentq`), or a straight-line objective with straight-line limits (`linprog`)?

In [ ]:
import numpy as np
from scipy import optimize

def f(v):
    x, y = v
    return (1 - x) ** 2 + 5 * (y - x ** 2) ** 2

res = optimize.minimize(f, x0=[-1.2, 1.0], method="BFGS")
print("success:", res.success)
print("x      :", res.x.round(4))
print("f(x)   :", float(res.fun) < 1e-8)
print("iterations:", res.nit, " function calls:", res.nfev)
print(res.message)

success: True
x      : [1. 1.]
f(x)   : True
iterations: 15  function calls: 57
Optimization terminated successfully.


In [ ]:
import numpy as np
from scipy import optimize

obj = lambda v: v[0] ** 2 + v[1] ** 2
cons = optimize.LinearConstraint([[1, 1]], lb=1, ub=np.inf)
bnds = optimize.Bounds([0, 0], [2, 2])
res = optimize.minimize(obj, x0=[2.0, 0.0], method="SLSQP", bounds=bnds, constraints=[cons])
print(res.x.round(4), round(res.fun, 4), res.success)

[0.5 0.5] 0.5 True


In [ ]:
import numpy as np
from scipy import optimize, stats

def model(t, a, k, c):
    return a * np.exp(-k * t) + c

rng = np.random.default_rng(5)
t = np.linspace(0, 10, 30)
y = model(t, 4.0, 0.5, 1.0) + rng.normal(0, 0.15, t.size)

popt, pcov = optimize.curve_fit(model, t, y, p0=[1, 1, 0])
perr = np.sqrt(np.diag(pcov))
dof = t.size - len(popt)
tcrit = stats.t.ppf(0.975, dof)
for name, v, e in zip("a k c".split(), popt, perr):
    print("%s = %.3f +/- %.3f   95%% CI (%.3f, %.3f)" % (name, v, e, v - tcrit * e, v + tcrit * e))
resid = y - model(t, *popt)
print("residual std: %.3f (true noise 0.15)" % resid.std(ddof=len(popt)))

a = 3.954 +/- 0.091   95% CI (3.767, 4.141)
k = 0.457 +/- 0.025   95% CI (0.405, 0.508)
c = 0.908 +/- 0.049   95% CI (0.806, 1.009)
residual std: 0.130 (true noise 0.15)


In [ ]:
import numpy as np
from scipy import optimize

g = lambda x: x * np.exp(x) - 5
x1 = optimize.brentq(g, 0, 3)
print("brentq x =", round(x1, 6), " check:", round(x1 * np.exp(x1), 6))

F = lambda v: [v[0] ** 2 + v[1] ** 2 - 4, v[1] - v[0]]
sol = optimize.root(F, [1.0, 1.0])
print("root:", sol.x.round(6), sol.success, " sqrt(2) =", round(np.sqrt(2), 6))

brentq x = 1.326725  check: 5.0
root: [1.414214 1.414214] True  sqrt(2) = 1.414214


In [ ]:
from scipy import optimize

c = [-12, -9]
A_ub = [[1, 1],
        [3, 1]]
b_ub = [40, 90]
res = optimize.linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, 25), (0, None)], method="highs")
print(res.status, res.message)
print("hours A, B:", res.x.round(2))
print("profit:", round(-res.fun, 2))
print("slack (unused) per limit:", res.slack.round(2))

0 Optimization terminated successfully. (HiGHS Status 7: Optimal)
hours A, B: [25. 15.]
profit: 435.0
slack (unused) per limit: [0. 0.]


## 18.5 scipy.integrate and scipy.interpolate

**Integration** adds up a quantity that changes continuously: area under a curve, total distance from a speed, probability from a density. **Interpolation** estimates values *between* known points. Both turn scattered or sampled numbers into continuous functions you can query anywhere.

In [ ]:
import numpy as np
from scipy import integrate, stats

area, err = integrate.quad(stats.norm.pdf, -1.96, 1.96)
print("quad area = %.6f  (error estimate %.1e)" % (area, err))
print("cdf diff  = %.6f" % (stats.norm.cdf(1.96) - stats.norm.cdf(-1.96)))

val, _ = integrate.quad(lambda x: x ** 2, 0, 3)
print("integral of x^2 on [0,3]:", round(val, 6), " mean value:", round(val / 3, 6))

tail, _ = integrate.quad(lambda x: np.exp(-x), 0, np.inf)
print("integral of exp(-x) to infinity:", round(tail, 6))

quad area = 0.950004  (error estimate 1.0e-11)
cdf diff  = 0.950004
integral of x^2 on [0,3]: 9.0  mean value: 3.0
integral of exp(-x) to infinity: 1.0


In [ ]:
import numpy as np
from scipy import integrate

t = np.linspace(0, 1, 11)                      # hours
speed = np.array([0, 20, 35, 42, 45, 40, 38, 30, 22, 10, 0.0])
print("trapezoid:", round(integrate.trapezoid(speed, t), 3), "km")
print("simpson  :", round(integrate.simpson(speed, x=t), 3), "km")
dist = integrate.cumulative_trapezoid(speed, t, initial=0)
print("distance at each stop:", dist.round(1))

trapezoid: 28.2 km
simpson  : 28.267 km
distance at each stop: [ 0.   1.   3.8  7.6 12.  16.2 20.1 23.5 26.1 27.7 28.2]


In [ ]:
import numpy as np
from scipy import integrate

r, K = 0.8, 1000.0
rhs = lambda t, N: r * N * (1 - N / K)
sol = integrate.solve_ivp(rhs, (0, 10), [50.0], t_eval=np.linspace(0, 10, 101), rtol=1e-8)
print("N(10) = %.1f" % sol.y[0, -1])
exact = K / (1 + (K / 50 - 1) * np.exp(-r * 10))
print("exact = %.1f" % exact)
cross = sol.t[np.argmax(sol.y[0] > 500)]
print("passes 500 users at about week", cross)

N(10) = 993.7
exact = 993.7
passes 500 users at about week 3.7


In [ ]:
import numpy as np
from scipy import interpolate

xk = np.array([0, 1, 2, 3, 4, 5, 6.0])
yk = np.array([0, 0.8, 0.9, 0.1, -0.8, -1.0, -0.3])

print("linear:", round(float(np.interp(2.5, xk, yk)), 4))
cs = interpolate.CubicSpline(xk, yk)
pc = interpolate.PchipInterpolator(xk, yk)
print("cubic :", round(float(cs(2.5)), 4))
print("pchip :", round(float(pc(2.5)), 4))
print("slope at 2.5:", round(float(cs.derivative()(2.5)), 4))
print("area 0..6   :", round(float(cs.integrate(0, 6)), 4))
print("zero crossings:", cs.roots(extrapolate=False).round(3))

linear: 0.5
cubic : 0.5768
pchip : 0.6059
slope at 2.5: -0.8452
area 0..6   : -0.1643
zero crossings: [0.    3.098]


## 18.6 scipy.linalg and scipy.sparse

**Linear algebra** solves systems like A x = b, finds the best-fitting line (least squares) and splits a matrix into simpler parts (SVD, eigenvalues). **Sparse matrices** store only the non-zero entries, so a matrix of 10 million cells that is 99.99% zero takes a few kilobytes. `numpy.linalg` covers the basics; `scipy.linalg` is a superset with more decompositions and options, and always uses optimised LAPACK.

In [ ]:
import numpy as np
from scipy import linalg

A = np.array([[2.0, 1.0, 1.0],
              [1.0, 3.0, 2.0],
              [1.0, 0.0, 4.0]])
b = np.array([100.0, 120.0, 80.0])
x = linalg.solve(A, b)
print("x       :", x.round(4))
print("A @ x   :", (A @ x).round(6))
print("det, cond:", round(linalg.det(A), 3), round(np.linalg.cond(A), 2))

B = np.array([[1.0, 2.0], [2.0, 4.0000001]])
print("cond of nearly singular B: %.2e" % np.linalg.cond(B))

x       : [33.6842 21.0526 11.5789]
A @ x   : [100. 120.  80.]
det, cond: 19.0 3.88
cond of nearly singular B: 2.50e+08


In [ ]:
import numpy as np
from scipy import linalg

rng = np.random.default_rng(0)
size = rng.uniform(40, 160, 200)
rooms = rng.integers(1, 6, 200)
price = 20 + 3.0 * size + 12 * rooms + rng.normal(0, 10, 200)
X = np.column_stack([np.ones(200), size, rooms])
w, res, rank, sv = linalg.lstsq(X, price)
print("weights:", w.round(2), " rank:", rank)

base = rng.normal(size=(200, 3))
M = np.column_stack([base, base[:, 0] + base[:, 1], 2 * base[:, 2]])   # 5 columns, 3 independent
s = linalg.svdvals(M)
print("singular values:", s.round(2))
print("numerical rank :", int((s > 1e-10 * s[0]).sum()))

weights: [23.4   2.99 11.27]  rank: 3
singular values: [30.55 25.31 12.47  0.    0.  ]
numerical rank : 3


In [ ]:
import numpy as np
from scipy import sparse

rng = np.random.default_rng(1)
n_users, n_items, per_user = 20000, 5000, 10
rows = np.repeat(np.arange(n_users), per_user)
cols = rng.integers(0, n_items, n_users * per_user)
vals = np.ones(rows.size)
M = sparse.coo_array((vals, (rows, cols)), shape=(n_users, n_items)).tocsr()

dense_mb = n_users * n_items * 8 / 1e6
sparse_mb = (M.data.nbytes + M.indices.nbytes + M.indptr.nbytes) / 1e6
print("dense  needs %.0f MB" % dense_mb)
print("sparse needs %.1f MB" % sparse_mb)
print("stored non-zeros:", M.nnz, " density: %.4f" % (M.nnz / (n_users * n_items)))

popularity = M.sum(axis=0)                 # purchases per product
print("most popular product id:", int(np.argmax(popularity)), "with", int(popularity.max()))
user_scores = M @ rng.normal(size=n_items)
print("scores shape:", user_scores.shape)

dense  needs 800 MB
sparse needs 3.4 MB
stored non-zeros: 199815  density: 0.0020
most popular product id: 2622 with 62
scores shape: (20000,)


In [ ]:
import numpy as np
from scipy import sparse
from scipy.sparse import linalg as spla
from scipy.sparse import csgraph

n = 100_000
A = sparse.diags_array([-np.ones(n - 1), 2 * np.ones(n), -np.ones(n - 1)], offsets=[-1, 0, 1], format="csc")
b = np.ones(n)
x = spla.spsolve(A, b)
print("residual max:", float(np.abs(A @ x - b).max()) < 1e-6, " x[n//2] = %.0f" % x[n // 2])

# roads: 0-1 (4 km), 1-2 (3), 0-2 (9), 3-4 (2); nodes 3,4 are a separate island
rows, cols, km = [0, 1, 0, 3], [1, 2, 2, 4], [4, 3, 9, 2]
G = sparse.coo_array((km, (rows, cols)), shape=(5, 5)).tocsr()
n_comp, labels = csgraph.connected_components(G, directed=False)
print("components:", n_comp, labels)
print("distance from 0:", csgraph.dijkstra(G, directed=False, indices=0))

residual max: True  x[n//2] = 1250025000
components: 2 [0 0 0 1 1]
distance from 0: [ 0.  4.  7. inf inf]


## 18.7 scipy.signal and scipy.fft

A **signal** is any measurement taken over time: sales per day, a heartbeat, a machine's vibration. Two ideas cover most uses. **Filtering** removes unwanted parts (noise, slow drift). The **Fourier transform** re-expresses the signal as a sum of waves, so you can see *which frequencies* it contains. Frequencies need a **sampling rate** `fs` (samples per second, or per day for daily data).

In [ ]:
import numpy as np
from scipy import fft

rng = np.random.default_rng(2)
fs = 200
t = np.arange(0, 2, 1 / fs)
x = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 30 * t) + rng.normal(0, 0.3, t.size)

amp = np.abs(fft.rfft(x)) * 2 / t.size
freq = fft.rfftfreq(t.size, 1 / fs)
top = np.argsort(amp)[-2:][::-1]
for i in top:
    print("peak at %.1f Hz, amplitude %.2f" % (freq[i], amp[i]))
print("bin spacing:", freq[1], "Hz; highest frequency:", freq[-1], "Hz (Nyquist = fs/2)")

peak at 5.0 Hz, amplitude 0.97
peak at 30.0 Hz, amplitude 0.47
bin spacing: 0.5 Hz; highest frequency: 100.0 Hz (Nyquist = fs/2)


In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(2)
fs = 200
t = np.arange(0, 2, 1 / fs)
clean = np.sin(2 * np.pi * 5 * t)
noisy = clean + rng.normal(0, 0.5, t.size)

sos = signal.butter(4, 10, btype="low", fs=fs, output="sos")
smooth = signal.sosfiltfilt(sos, noisy)
print("noise rms before: %.3f" % np.sqrt(np.mean((noisy - clean) ** 2)))
print("noise rms after : %.3f" % np.sqrt(np.mean((smooth - clean) ** 2)))

noise rms before: 0.504
noise rms after : 0.125


In [ ]:
import numpy as np
from scipy import signal, fft

rng = np.random.default_rng(8)
days = np.arange(140)
orders = 100 + 20 * np.sin(2 * np.pi * days / 7) + rng.normal(0, 4, days.size)

smooth = signal.savgol_filter(orders, window_length=5, polyorder=2)
peaks, props = signal.find_peaks(smooth, distance=5, prominence=15)
print("peak days:", peaks[:6], "... total", len(peaks))
print("gaps between peaks:", np.diff(peaks)[:8])

spec = np.abs(fft.rfft(orders - orders.mean()))
f = fft.rfftfreq(days.size, d=1.0)             # cycles per day
print("dominant period: %.1f days" % (1 / f[np.argmax(spec)]))

peak days: [ 2  9 16 23 30 36] ... total 20
gaps between peaks: [7 7 7 7 6 8 7 7]
dominant period: 7.0 days


## 18.8 scipy.spatial and scipy.cluster

**Spatial** tools answer "who is near whom?". A **KDTree** splits space into boxes once so that each nearest-neighbour query skips most of the data. **Distance functions** measure similarity between rows. **Clustering** groups points that belong together; `scipy.cluster` offers k-means and hierarchical clustering (for the full machine-learning treatment see scikit-learn and the Machine Learning document).

In [ ]:
import time
import numpy as np
from scipy import spatial

rng = np.random.default_rng(0)
shops = rng.uniform(0, 100, (200_000, 2))
custs = rng.uniform(0, 100, (2_000, 2))

t0 = time.perf_counter()
tree = spatial.KDTree(shops)
dist, idx = tree.query(custs, k=3)
t_tree = time.perf_counter() - t0

t0 = time.perf_counter()
brute = np.array([np.argsort(((shops - c) ** 2).sum(axis=1))[:3] for c in custs[:200]])
t_brute = (time.perf_counter() - t0) * 10          # scaled up from 200 to 2000 customers

print("same answer:", np.array_equal(brute, idx[:200]))
print("tree at least 10x faster:", t_brute > 10 * t_tree)
print("first customer, nearest shop distance: %.3f" % dist[0, 0])

same answer: True
tree at least 10x faster: True
first customer, nearest shop distance: 0.149


In [ ]:
import numpy as np
from scipy.spatial import distance
from scipy.cluster import hierarchy

X = np.array([[1.0, 2], [1.2, 2.5], [0.8, 1.8],
              [8.0, 9], [8.5, 9.4], [7.6, 8.8]])
D = distance.squareform(distance.pdist(X))
print("distance 0-1: %.3f   0-3: %.3f" % (D[0, 1], D[0, 3]))
Z = hierarchy.linkage(X, method="ward")
labels = hierarchy.fcluster(Z, t=2, criterion="maxclust")
print("labels:", labels)
print("merge heights:", Z[:, 2].round(2))
print("cosine distance of rows 0 and 3: %.4f" % distance.cosine(X[0], X[3]))

distance 0-1: 0.539   0-3: 9.899
labels: [1 1 1 2 2 2]
merge heights: [ 0.28  0.45  0.77  0.99 17.15]
cosine distance of rows 0 and 3: 0.0344


In [ ]:
import numpy as np
from scipy.cluster import vq

rng = np.random.default_rng(4)
pts = np.vstack([rng.normal(c, 0.6, (60, 2)) for c in ([0, 0], [4, 1], [2, 4.5])])
centres, labels = vq.kmeans2(pts, 3, seed=1, minit="++")
print("sizes:", np.bincount(labels))
print("centres:\n", centres[np.argsort(centres[:, 0])].round(2))
new_point = np.array([[3.8, 0.7]])
code, dist = vq.vq(new_point, centres)
print("new point joins centre", int(code[0]), "at distance %.2f" % dist[0])

sizes: [60 60 60]
centres:
 [[-0.11  0.07]
 [ 2.2   4.51]
 [ 4.1   1.04]]
new point joins centre 2 at distance 0.45


## 18.9 statsmodels: OLS, GLM, ARIMA and the Formula API

scikit-learn asks "how well does this predict?". statsmodels asks "what do the coefficients mean, and how sure are we?". It returns standard errors, t-statistics, p-values, confidence intervals and diagnostic tests in one summary table, which is what analysts, economists and scientists must report. **statsmodels is not installed on this study machine.** Each statsmodels block below is labelled *not run here*: it is standard, idiomatic code, but no output is shown for it, because I will not invent output. Directly after each one comes a NumPy/SciPy version that did run, which computes the same quantities from first principles. Run the statsmodels code yourself after `python3 -m pip install statsmodels` and compare.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(11)
x = rng.uniform(0, 10, 80)
y = 2 + 0.7 * x + rng.normal(0, 1.0, 80)

X = np.column_stack([np.ones_like(x), x])
n, k = X.shape
beta = np.linalg.solve(X.T @ X, X.T @ y)              # normal equations
resid = y - X @ beta
sigma2 = resid @ resid / (n - k)                      # unbiased noise variance
cov = sigma2 * np.linalg.inv(X.T @ X)
se = np.sqrt(np.diag(cov))
t = beta / se
p = 2 * stats.t.sf(np.abs(t), df=n - k)
tcrit = stats.t.ppf(0.975, n - k)
r2 = 1 - resid @ resid / ((y - y.mean()) @ (y - y.mean()))

for name, b, s, tv, pv in zip(["const", "x"], beta, se, t, p):
    print("%-5s coef=%.4f se=%.4f t=%.2f p=%.2g CI=(%.4f, %.4f)" % (name, b, s, tv, pv, b - tcrit * s, b + tcrit * s))
print("R-squared = %.4f" % r2)

const coef=2.0100 se=0.2054 t=9.78 p=3.3e-15 CI=(1.6010, 2.4190)
x     coef=0.7166 se=0.0391 t=18.35 p=5.1e-30 CI=(0.6389, 0.7944)
R-squared = 0.8119


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(11)
x = rng.uniform(0, 10, 80)
y = 2 + 0.7 * x + rng.normal(0, 0.4 + 0.12 * x)
X = np.column_stack([np.ones_like(x), x])
n, k = X.shape
XtX_inv = np.linalg.inv(X.T @ X)
beta = XtX_inv @ X.T @ y
resid = y - X @ beta

se_classic = np.sqrt(np.diag(resid @ resid / (n - k) * XtX_inv))
h = np.einsum("ij,jk,ik->i", X, XtX_inv, X)            # leverage of each row
w = resid / (1 - h)
meat = (X * w[:, None]).T @ (X * w[:, None])
se_hc3 = np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv))
print("slope %.3f" % beta[1])
print("classic se: %.4f   robust HC3 se: %.4f" % (se_classic[1], se_hc3[1]))

fit = X @ beta
lo = np.abs(resid[fit < np.median(fit)]).mean()
hi = np.abs(resid[fit >= np.median(fit)]).mean()
print("mean |residual| low half: %.2f, high half: %.2f" % (lo, hi))
print("Spearman(fitted, |resid|) = %.2f" % stats.spearmanr(fit, np.abs(resid)).statistic)

slope 0.719
classic se: 0.0406   robust HC3 se: 0.0486
mean |residual| low half: 0.51, high half: 1.08
Spearman(fitted, |resid|) = 0.47


In [ ]:
import numpy as np
from scipy import optimize, stats

rng = np.random.default_rng(0)
temp = rng.uniform(-5, 30, 300)
weekend = rng.integers(0, 2, 300)
tickets = rng.poisson(np.exp(1.0 + 0.04 * temp - 0.5 * weekend))
X = np.column_stack([np.ones(300), temp, weekend])

def nll(b):                                    # negative log-likelihood (constant dropped)
    eta = X @ b
    return np.sum(np.exp(eta) - tickets * eta)

res = optimize.minimize(nll, np.zeros(3), method="BFGS")
b = res.x
mu = np.exp(X @ b)
cov = np.linalg.inv(X.T @ (X * mu[:, None]))   # inverse Fisher information
se = np.sqrt(np.diag(cov))
z = b / se
for name, bi, si, zi in zip(["const", "temp", "weekend"], b, se, z):
    print("%-8s coef=%.3f se=%.3f z=%.1f p=%.3g" % (name, bi, si, zi, 2 * stats.norm.sf(abs(zi))))
print("rate ratio for weekend: %.2f" % np.exp(b[2]))

const    coef=1.024 se=0.066 z=15.5 p=3.2e-54
temp     coef=0.034 se=0.003 z=11.3 p=2.15e-29
weekend  coef=-0.418 se=0.059 z=-7.1 p=1.76e-12
rate ratio for weekend: 0.66


In [ ]:
import numpy as np

rng = np.random.default_rng(5)
y = np.zeros(300)
for i in range(1, 300):
    y[i] = 0.7 * y[i - 1] + rng.normal()

ybar = y.mean()
z = y - ybar
phi = (z[1:] @ z[:-1]) / (z[:-1] @ z[:-1])      # least-squares AR(1) coefficient
sigma = np.std(z[1:] - phi * z[:-1], ddof=1)
print("phi = %.3f  noise std = %.3f" % (phi, sigma))

forecast, last = [], z[-1]
for h in range(5):
    last = phi * last
    forecast.append(ybar + last)
print("forecast:", np.round(forecast, 3))
print("last observed value: %.3f, series mean: %.3f" % (y[-1], ybar))

phi = 0.715  noise std = 0.960
forecast: [-1.439 -1.067 -0.8   -0.61  -0.473]
last observed value: -1.960, series mean: -0.130


## 18.10 matplotlib Quick Reference for Plotting Results

This is only a cheat sheet for plotting the results in this chapter. The full visualisation guide (chart choice, design, storytelling) is in the Data Analytics document, Chapter 8. Always use the **object-oriented style**: `fig, ax = plt.subplots()`, then call methods on `ax`. It scales to many panels and avoids the confusion of hidden "current figure" state.

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(7)
data = rng.lognormal(1.2, 0.5, 400)
s, loc, scale = stats.lognorm.fit(data, floc=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
stats.probplot(np.log(data), dist="norm", plot=axes[0])
axes[0].set_title("Q-Q plot of log(data)")

axes[1].hist(data, bins=30, density=True, alpha=0.5)
xs = np.linspace(0, data.max(), 200)
axes[1].plot(xs, stats.lognorm.pdf(xs, s, loc, scale), lw=2)
axes[1].set_title("histogram + fitted pdf")

groups = [rng.normal(m, 2, 30) for m in (10, 11, 13)]
means = [g.mean() for g in groups]
errs = [1.96 * stats.sem(g) for g in groups]
axes[2].errorbar([0, 1, 2], means, yerr=errs, fmt="o", capsize=4)
axes[2].set_xticks([0, 1, 2], ["A", "B", "C"])
axes[2].set_title("means with 95% CI")

fig.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=80)
plt.close(fig)
print("panels:", len(axes), " png bytes > 5000:", buf.tell() > 5000)
print("x tick labels of last panel:", [t.get_text() for t in axes[2].get_xticklabels()])

panels: 3  png bytes > 5000: True
x tick labels of last panel: ['A', 'B', 'C']


## 18.11 Recipes: How Do I ...?

Each recipe is a complete, standalone answer to a real task. Copy it, change the data, and keep the verification line.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(10)
data = rng.gamma(shape=3.0, scale=2.0, size=500)

rows = []
for name in ["norm", "lognorm", "gamma", "weibull_min", "expon"]:
    dist = getattr(stats, name)
    params = dist.fit(data, floc=0) if name != "norm" else dist.fit(data)
    ll = dist.logpdf(data, *params).sum()
    aic = 2 * len(params) - 2 * ll
    rows.append((aic, name))
for aic, name in sorted(rows):
    print("%-12s AIC = %.1f" % (name, aic))

gamma        AIC = 2544.8
weibull_min  AIC = 2562.7
lognorm      AIC = 2565.1
norm         AIC = 2671.7
expon        AIC = 2795.8


In [ ]:
import numpy as np
from scipy import stats

conv_a, n_a = 120, 2000
conv_b, n_b = 156, 2000
pa, pb = conv_a / n_a, conv_b / n_b
pool = (conv_a + conv_b) / (n_a + n_b)
se = np.sqrt(pool * (1 - pool) * (1 / n_a + 1 / n_b))
z = (pb - pa) / se
print("rates %.3f vs %.3f  z = %.2f  p = %.4f" % (pa, pb, z, 2 * stats.norm.sf(abs(z))))

se_diff = np.sqrt(pa * (1 - pa) / n_a + pb * (1 - pb) / n_b)
lo, hi = (pb - pa) + np.array([-1, 1]) * stats.norm.ppf(0.975) * se_diff
print("lift 95%% CI: (%.4f, %.4f)" % (lo, hi))

# sample size per group to detect 6% -> 7.8% with 80% power, 5% two-sided
p1, p2 = 0.06, 0.078
zb = stats.norm.ppf(0.80) + stats.norm.ppf(0.975)
n = zb ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2
print("needed per group:", int(np.ceil(n)))

rates 0.060 vs 0.078  z = 2.25  p = 0.0247
lift 95% CI: (0.0023, 0.0337)
needed per group: 3109


In [ ]:
import numpy as np
from scipy import optimize, stats

def model(t, a, k):
    return a * (1 - np.exp(-k * t))

rng = np.random.default_rng(2)
t = np.linspace(0.2, 8, 25)
y = model(t, 10, 0.6) + rng.normal(0, 0.3, t.size)
p, cov = optimize.curve_fit(model, t, y, p0=[5, 1])

t_new = 6.0
eps = 1e-6
grad = np.array([(model(t_new, *(p + eps * np.eye(2)[i])) - model(t_new, *p)) / eps for i in range(2)])
se_mean = np.sqrt(grad @ cov @ grad)                    # delta method
s_resid = np.sqrt(np.sum((y - model(t, *p)) ** 2) / (t.size - 2))
tc = stats.t.ppf(0.975, t.size - 2)
fit = model(t_new, *p)
print("fit at t=6: %.2f" % fit)
print("95%% CI of the curve : (%.2f, %.2f)" % (fit - tc * se_mean, fit + tc * se_mean))
print("95%% prediction range: (%.2f, %.2f)" % (fit - tc * np.hypot(se_mean, s_resid), fit + tc * np.hypot(se_mean, s_resid)))

fit at t=6: 9.73
95% CI of the curve : (9.58, 9.87)
95% prediction range: (9.16, 10.29)


In [ ]:
import numpy as np
from scipy import optimize

value = np.array([60, 100, 120, 80, 30.0])        # expected profit per campaign
cost = np.array([10, 20, 30, 25, 5.0])            # budget needed
budget = 55
res = optimize.milp(c=-value,
                    constraints=optimize.LinearConstraint(cost[None, :], ub=budget),
                    integrality=np.ones(5), bounds=optimize.Bounds(0, 1))
chosen = np.round(res.x).astype(int)
print("chosen campaigns:", chosen, " profit:", int(value @ chosen), " cost:", int(cost @ chosen))

chosen campaigns: [0 1 1 0 1]  profit: 250  cost: 55


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(6)
x = rng.normal(size=40)
y = 0.4 * x + rng.normal(size=40)

stat = lambda a, b: np.corrcoef(a, b)[0, 1]
perm = stats.permutation_test((x, y), stat, permutation_type="pairings",
                              n_resamples=5000, random_state=1)
print("r = %.3f  permutation p = %.4f  (pearsonr p = %.4f)" % (perm.statistic, perm.pvalue, stats.pearsonr(x, y).pvalue))

boot = stats.bootstrap((x, y), lambda a, b: np.corrcoef(a, b)[0, 1], paired=True,
                       vectorized=False, n_resamples=2000, random_state=1)
print("bootstrap 95%% CI for r: (%.3f, %.3f)" % (boot.confidence_interval.low, boot.confidence_interval.high))

r = 0.470  permutation p = 0.0008  (pearsonr p = 0.0022)
bootstrap 95% CI for r: (0.184, 0.673)


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(3)
x = np.append(rng.normal(100, 5, 200), [160, 40])
z = stats.zscore(x)
mad = stats.median_abs_deviation(x, scale="normal")
robust_z = (x - np.median(x)) / mad
print("classic |z|>3 count:", int((np.abs(z) > 3).sum()))
print("robust  |z|>3 count:", int((np.abs(robust_z) > 3).sum()))
print("flagged values:", np.sort(x[np.abs(robust_z) > 3]).round(1))

classic |z|>3 count: 2
robust  |z|>3 count: 3
flagged values: [ 40.  116.6 160. ]


In [ ]:
import numpy as np
import pandas as pd
from scipy import interpolate

s = pd.Series([10.0, 12.0, np.nan, np.nan, 18.0, 17.0, np.nan, 14.0],
              index=pd.date_range("2025-01-01", periods=8, freq="D"))
known = s.dropna()
xk = (known.index - s.index[0]).days.to_numpy()
pchip = interpolate.PchipInterpolator(xk, known.to_numpy())
filled = pd.Series(pchip((s.index - s.index[0]).days), index=s.index).round(2)
print(filled.tolist())
print("pandas linear:", s.interpolate().round(2).tolist())

[10.0, 12.0, 14.44, 16.89, 18.0, 17.0, 15.66, 14.0]
pandas linear: [10.0, 12.0, 14.0, 16.0, 18.0, 17.0, 15.5, 14.0]


In [ ]:
import numpy as np
from scipy import optimize
from scipy.spatial import distance

drivers = np.array([[0, 0], [5, 5], [9, 1.0]])
orders = np.array([[8, 2], [1, 1], [6, 6.0]])
cost = distance.cdist(drivers, orders)
rows, cols = optimize.linear_sum_assignment(cost)
print("driver -> order:", list(zip(rows.tolist(), cols.tolist())))
print("total distance: %.2f" % cost[rows, cols].sum())
print("greedy first choice would pair driver 0 with order", int(cost[0].argmin()))

driver -> order: [(0, 1), (1, 2), (2, 0)]
total distance: 4.24
greedy first choice would pair driver 0 with order 1


In [ ]:
import numpy as np
from scipy.spatial import KDTree

def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

wh = {"Rotterdam": (51.92, 4.48), "Berlin": (52.52, 13.40), "Madrid": (40.42, -3.70), "Milan": (45.46, 9.19)}
names = list(wh)
tree = KDTree(to_xyz(*np.array(list(wh.values())).T))
d, i = tree.query(to_xyz(48.86, 2.35))                 # Paris
print("nearest to Paris:", names[int(i[0])])
print("chord distance %.4f -> about %.0f km" % (d[0], 6371 * 2 * np.arcsin(d[0] / 2)))

nearest to Paris: Rotterdam
chord distance 0.0584 -> about 372 km


In [ ]:
import numpy as np
from scipy import optimize

price = lambda q: 50 * np.exp(-0.01 * q)          # price falls as quantity rises
profit = lambda q: q * price(q) - (400 + 12 * q)
low = optimize.root_scalar(profit, bracket=[1, 20])
best = optimize.minimize_scalar(lambda q: -profit(q), bounds=(1, 200), method="bounded")
print("break-even at q = %.2f" % low.root)
print("max profit %.1f at q = %.1f" % (-best.fun, best.x))

break-even at q = 12.44
max profit 527.8 at q = 57.4


In [ ]:
import numpy as np
from scipy import integrate

fpr = np.array([0, 0.1, 0.3, 0.6, 1.0])
tpr = np.array([0, 0.5, 0.75, 0.9, 1.0])
print("AUC by trapezoid: %.4f" % integrate.trapezoid(tpr, fpr))
print("random-guess AUC: %.4f" % integrate.trapezoid(fpr, fpr))

AUC by trapezoid: 0.7775
random-guess AUC: 0.5000


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(9)
df = pd.DataFrame({"a": rng.normal(size=60)})
df["b"] = df["a"] * 0.8 + rng.normal(size=60)
df["c"] = rng.normal(size=60)
res = stats.spearmanr(df)                      # a table in, a matrix out
print(pd.DataFrame(res.statistic, index=df.columns, columns=df.columns).round(2))
print("p(a,b) = %.2g   p(a,c) = %.2f" % (res.pvalue[0, 1], res.pvalue[0, 2]))

a     b     c
a  1.00  0.54 -0.19
b  0.54  1.00 -0.12
c -0.19 -0.12  1.00
p(a,b) = 8.7e-06   p(a,c) = 0.14


## 18.12 Performance and Pitfalls

In [ ]:
import time
import numpy as np
from scipy import stats

x = np.random.default_rng(0).normal(size=20000)
t0 = time.perf_counter()
slow = [stats.norm.cdf(v) for v in x]
t_loop = time.perf_counter() - t0
t0 = time.perf_counter()
fast = stats.norm.cdf(x)
t_vec = time.perf_counter() - t0
print("same values:", np.allclose(slow, fast))
print("vectorised at least 20x faster:", t_loop > 20 * t_vec)

same values: True
vectorised at least 20x faster: True


In [ ]:
from scipy import stats
d = stats.poisson(4)
print("wrong  sf(10):", round(d.sf(10), 5), "  right sf(9):", round(d.sf(9), 5), "  by pmf sum:", round(sum(d.pmf(k) for k in range(10, 60)), 5))

wrong  sf(10): 0.00284   right sf(9): 0.00813   by pmf sum: 0.00813


In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(0)
a = rng.normal(0, 1, 10); b = rng.normal(0, 6, 100)
print("student p = %.3f   welch p = %.3f" % (stats.ttest_ind(a, b).pvalue, stats.ttest_ind(a, b, equal_var=False).pvalue))

student p = 0.906   welch p = 0.729


In [ ]:
import numpy as np
from scipy import optimize
rng = np.random.default_rng(0)
t = np.linspace(0, 10, 100); y = 2 * np.sin(3 * t) + rng.normal(0, 0.1, 100)
f = lambda t, a, w: a * np.sin(w * t)
for p0 in (None, [1, 2.9]):
    p, _ = optimize.curve_fit(f, t, y, p0=p0)
    print("p0=%-10s params=%s rmse=%.3f" % (p0, p.round(2), np.sqrt(np.mean((y - f(t, *p)) ** 2))))

p0=None       params=[0.05 1.02] rmse=1.442
p0=[1, 2.9]   params=[2.03 3.  ] rmse=0.094


In [ ]:
import numpy as np
from scipy import stats
d = np.random.default_rng(0).lognormal(1.2, 0.5, 200)
print("free loc  :", np.round(stats.lognorm.fit(d), 3))
print("floc=0    :", np.round(stats.lognorm.fit(d, floc=0), 3))

free loc  : [ 0.434 -0.322  3.703]
floc=0    : [0.481 0.    3.346]


In [ ]:
import numpy as np
x = np.array([3, 1, 2.0]); y = np.array([30, 10, 20.0])
print("wrong:", np.interp(1.5, x, y), "  right:", np.interp(1.5, *zip(*sorted(zip(x, y)))))

wrong: 30.0   right: 15.0


In [ ]:
import numpy as np
from scipy import sparse
d = np.array([[1, 2], [3, 4]])
print("csr_matrix A*A:", (sparse.csr_matrix(d) * sparse.csr_matrix(d)).toarray().tolist())
print("csr_array  A*A:", (sparse.csr_array(d) * sparse.csr_array(d)).toarray().tolist())
print("csr_array  A@A:", (sparse.csr_array(d) @ sparse.csr_array(d)).toarray().tolist())

csr_matrix A*A: [[7, 10], [15, 22]]
csr_array  A*A: [[1, 4], [9, 16]]
csr_array  A@A: [[7, 10], [15, 22]]


In [ ]:
import numpy as np
from scipy import linalg
H = linalg.hilbert(10); truth = np.ones(10); b = H @ truth
err_inv = np.abs(np.linalg.inv(H) @ b - truth).max()
err_solve = np.abs(linalg.solve(H, b) - truth).max()
print("inverse error is larger than solve error:", err_inv > err_solve)

inverse error is larger than solve error: True


In [ ]:
import numpy as np
from scipy import fft
fs = 200; t = np.arange(0, 2, 1 / fs); x = np.sin(2 * np.pi * 5 * t)
k = np.argmax(np.abs(fft.rfft(x)))
print("wrong:", fft.rfftfreq(t.size)[k], "  right:", fft.rfftfreq(t.size, 1 / fs)[k], "Hz")

wrong: 0.025   right: 5.0 Hz


In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(0)
a = rng.normal(0, 1, 200000); b = rng.normal(0.01, 1, 200000)
r = stats.ttest_ind(a, b)
print("p = %.4f   difference = %.4f standard deviations" % (r.pvalue, abs(a.mean() - b.mean())))

p = 0.0015   difference = 0.0101 standard deviations


In [ ]:
import numpy as np
from scipy import stats
a = stats.norm.rvs(size=3, random_state=42); b = stats.norm.rvs(size=3, random_state=42)
c = stats.norm.rvs(size=3); d = stats.norm.rvs(size=3)
print("seeded runs identical:", np.array_equal(a, b), "  unseeded identical:", np.array_equal(c, d))

seeded runs identical: True   unseeded identical: False


In [ ]:
import pandas as pd
df = pd.DataFrame({"x": [1.0, 2.0, 3.0]})
a = df["x"].to_numpy()
try:
    a[0] = 9
except ValueError as e:
    print("error:", e)
a = df["x"].to_numpy().copy(); a[0] = 9
print("copy edited:", a.tolist(), " table unchanged:", df["x"].tolist())

error: assignment destination is read-only
copy edited: [9.0, 2.0, 3.0]  table unchanged: [1.0, 2.0, 3.0]


In [ ]:
import numpy as np
from scipy.cluster import vq
rng = np.random.default_rng(0)
age = np.concatenate([rng.normal(25, 3, 100), rng.normal(55, 3, 100)])   # two clear age groups
income = rng.normal(50000, 15000, 200)                                      # unrelated, huge scale
X = np.column_stack([age, income])
for name, data in [("raw", X), ("whitened", vq.whiten(X))]:
    _, lab = vq.kmeans2(data, 2, seed=1, minit="++")
    truth = np.arange(200) >= 100
    agree = max(np.mean(lab == truth), np.mean(lab != truth))
    print("%-9s agreement with the age groups: %.2f" % (name, agree))

raw       agreement with the age groups: 0.56
whitened  agreement with the age groups: 1.00


## 18.13 Cross-Library Integration

The libraries share NumPy arrays, so integration is mostly about moving data cleanly. Three patterns cover most real work.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(12)
df = pd.DataFrame({
    "city": np.repeat(["Oslo", "Lima", "Pune"], 60),
    "variant": np.tile(["control", "new"], 90),
    "minutes": np.concatenate([rng.normal(m, 6, 60) for m in (30, 34, 28)]),
})
df.loc[df["variant"] == "new", "minutes"] -= 3.0

rows = []
for city, g in df.groupby("city"):
    a = g.loc[g["variant"] == "control", "minutes"]
    b = g.loc[g["variant"] == "new", "minutes"]
    r = stats.ttest_ind(b, a, equal_var=False)
    rows.append({"city": city, "diff": b.mean() - a.mean(), "p": r.pvalue})
out = pd.DataFrame(rows)
out["p_adj"] = stats.false_discovery_control(out["p"])
print(out.round(3).to_string(index=False))
print("city dtype:", df["city"].dtype)

city   diff     p  p_adj
Lima -3.415 0.021  0.036
Oslo -3.383 0.024  0.036
Pune -2.228 0.157  0.157
city dtype: str


In [ ]:
import numpy as np
from scipy import stats, sparse
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV

rng = np.random.default_rng(0)
X = sparse.random_array((400, 50), density=0.1, rng=rng, format="csr")
w = rng.normal(size=50)
y = X @ w + rng.normal(0, 0.1, 400)

search = RandomizedSearchCV(Ridge(), {"alpha": stats.loguniform(1e-3, 1e2)},
                            n_iter=12, cv=3, random_state=0)
search.fit(X, y)
print("input is sparse:", sparse.issparse(X), " nnz =", X.nnz)
print("best alpha: %.4f   cv R2: %.3f" % (search.best_params_["alpha"], search.best_score_))

input is sparse: True  nnz = 2000
best alpha: 0.0826   cv R2: 0.988


In [ ]:
import numpy as np
from joblib import Parallel, delayed

data = np.random.default_rng(0).lognormal(1.2, 0.5, 300)

def boot_median(seed):
    r = np.random.default_rng(seed)
    return np.median(r.choice(data, size=data.size, replace=True))

meds = Parallel(n_jobs=2)(delayed(boot_median)(s) for s in range(400))
print("n resamples:", len(meds), " 95%% interval: (%.3f, %.3f)" % tuple(np.percentile(meds, [2.5, 97.5])))
again = Parallel(n_jobs=2)(delayed(boot_median)(s) for s in range(400))
print("reproducible:", meds == again)

n resamples: 400  95% interval: (2.990, 3.467)
reproducible: True


## 18.14 How to Solve Any SciPy/statsmodels Problem

In [ ]:
from scipy import stats
# A server answers in a lognormal time with median 200 ms and shape 0.4. P(slower than 500 ms)?
d = stats.lognorm(s=0.4, scale=200)
print("P(>500 ms) = %.4f   99th percentile = %.0f ms" % (d.sf(500), d.ppf(0.99)))

P(>500 ms) = 0.0110   99th percentile = 507 ms


In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(21)
before = rng.lognormal(3.0, 0.5, 40)
after = before * rng.lognormal(-0.15, 0.2, 40)          # same users measured twice
diff = np.log(after) - np.log(before)
print("normal log-differences? shapiro p = %.2f" % stats.shapiro(diff).pvalue)
t = stats.ttest_1samp(diff, 0.0)
ci = t.confidence_interval()
print("paired on log scale: p = %.2g, mean ratio %.2f (CI %.2f to %.2f)" % (t.pvalue, np.exp(diff.mean()), np.exp(ci.low), np.exp(ci.high)))

normal log-differences? shapiro p = 0.46
paired on log scale: p = 5.4e-08, mean ratio 0.85 (CI 0.81 to 0.89)


In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(4)
x = rng.exponential(5, 50)
r = stats.bootstrap((x,), lambda s: np.percentile(s, 90), n_resamples=3000, random_state=0)
print("p90 = %.2f, 95%% CI (%.2f, %.2f)" % (np.percentile(x, 90), *r.confidence_interval))

p90 = 13.91, 95% CI (9.24, 23.30)


In [ ]:
import numpy as np
from scipy import optimize
# Split 100 units between 3 channels with diminishing returns a*sqrt(x); maximise total return.
a = np.array([3.0, 2.0, 1.0])
res = optimize.minimize(lambda x: -(a * np.sqrt(x)).sum(), x0=[30, 30, 40], bounds=[(0, 100)] * 3,
                        constraints=[{"type": "eq", "fun": lambda x: x.sum() - 100}], method="SLSQP")
print(res.x.round(1), "return %.2f" % -res.fun)
print("theory (x proportional to a^2):", (100 * a ** 2 / (a ** 2).sum()).round(1))

[64.3 28.6  7.1] return 37.42
theory (x proportional to a^2): [64.3 28.6  7.1]


In [ ]:
import numpy as np
from scipy import integrate
# Drug concentration decays at rate 0.3/h; a 10-unit dose is given each 4 h in a continuous 'pump' approximation.
sol = integrate.solve_ivp(lambda t, c: 10 / 4 - 0.3 * c, (0, 24), [0.0], t_eval=[24], rtol=1e-8)
print("c(24h) = %.3f   steady state = %.3f" % (sol.y[0, 0], 10 / 4 / 0.3))

c(24h) = 8.327   steady state = 8.333


In [ ]:
import numpy as np
from scipy import fft, spatial
rng = np.random.default_rng(1)
sales = 50 + 8 * np.sin(2 * np.pi * np.arange(364) / 52) + rng.normal(0, 2, 364)   # daily data with a 52-day cycle
f = fft.rfftfreq(364, 1.0)
print("cycle length: %.0f days" % (1 / f[1:][np.argmax(np.abs(fft.rfft(sales - sales.mean()))[1:])]))
pts = rng.uniform(size=(1000, 2))
print("pairs closer than 0.02:", len(spatial.KDTree(pts).query_pairs(0.02)))

cycle length: 52 days
pairs closer than 0.02: 612


## 18.15 Going Deeper: Internals, Memory, Speed, Interviews

In [ ]:
import numpy as np
from scipy import stats

class Triangle(stats.rv_continuous):
    "Density 2x on [0, 1]"
    def _pdf(self, x):
        return 2 * x

tri = Triangle(a=0, b=1, name="tri")
print("cdf(0.5) = %.4f   (exact 0.25)" % tri.cdf(0.5))
print("ppf(0.25) = %.4f  (exact 0.5)" % tri.ppf(0.25))
print("mean = %.4f   (exact 2/3)" % tri.mean())
print("sample mean = %.3f" % tri.rvs(size=2000, random_state=1).mean())

cdf(0.5) = 0.2500   (exact 0.25)
ppf(0.25) = 0.5000  (exact 0.5)
mean = 0.6667   (exact 2/3)
sample mean = 0.673


In [ ]:
import numpy as np
from scipy import sparse

rng = np.random.default_rng(0)
n = 5000
r = rng.integers(0, n, 50000); c = rng.integers(0, n, 50000)
coo = sparse.coo_array((np.ones(50000), (r, c)), shape=(n, n))
csr = coo.tocsr()
lil = coo.tolil()
mb = lambda b: round(b / 1e6, 2)
print("dense float64 :", mb(n * n * 8), "MB")
print("COO           :", mb(coo.data.nbytes + coo.row.nbytes + coo.col.nbytes), "MB")
print("CSR           :", mb(csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes), "MB")
print("index dtype   :", csr.indices.dtype, " nnz after duplicates summed:", csr.nnz)

dense float64 : 200.0 MB
COO           : 1.2 MB
CSR           : 0.84 MB
index dtype   : int64  nnz after duplicates summed: 49949


## Chapter recap

## Practice set

Twenty-five problems, graded from easy to expert. Try each before reading the answer.

In [ ]:
from scipy import stats
print(stats.norm.ppf(0.5, loc=10, scale=3))

In [ ]:
from scipy import optimize
res = optimize.minimize(lambda v: (v[0] - 3) ** 2 + 1, x0=[0.0])
print(round(float(res.x[0]), 3), round(float(res.fun), 3))

In [ ]:
import numpy as np
from scipy import sparse
A = sparse.csr_array(np.array([[0, 0, 3], [4, 0, 0], [0, 0, 0]]))
print(A.nnz, A.shape, A.toarray().sum())

In [ ]:
import numpy as np
print(np.interp([0, 1.5, 9], [1, 2], [10, 20]))

In [ ]:
from scipy import optimize
res = optimize.linprog([3, 2], A_ub=[[1, 1]], b_ub=[4], bounds=[(0, None)] * 2)
print(res.x, res.fun)

In [ ]:
import numpy as np
from scipy import integrate
t = np.array([0.0, 1.0, 2.0, 3.0]); v = np.array([5.0, 6.0, 7.0, 6.0])
print(integrate.trapezoid(t, v))

In [ ]:
from scipy import stats
d = stats.poisson(2)
print(round(d.cdf(2), 4), round(d.sf(2), 4), round(d.pmf(2), 4))

In [ ]:
import numpy as np
from scipy import stats

x = np.random.default_rng(0).gamma(2.0, 3.0, 80)
r = stats.bootstrap((x,), lambda s, axis: np.percentile(s, 75, axis=axis), n_resamples=3000,
                    confidence_level=0.90, random_state=0)
print("p75 = %.2f, 90%% CI (%.2f, %.2f)" % (np.percentile(x, 75), r.confidence_interval.low, r.confidence_interval.high))

p75 = 6.94, 90% CI (6.39, 8.32)


In [ ]:
import numpy as np
from scipy import stats
print(stats.zscore([1, 2, 3]).round(4), stats.zscore([1, 2, 3], ddof=1).round(4))

In [ ]:
import numpy as np
from scipy import optimize
print(round(optimize.brentq(lambda x: np.cos(x) - x, 0, 1), 6))

In [ ]:
import numpy as np
from scipy import signal
rng = np.random.default_rng(0)
x = 10 * np.sin(2 * np.pi * np.arange(140) / 7) + rng.normal(0, 4, 140)
print(len(signal.find_peaks(x)[0]), len(signal.find_peaks(x, prominence=10, distance=5)[0]))

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
x = rng.uniform(0, 10, 80)
y = 2 + 0.7 * x + rng.normal(0, 1.0, 80)
X = np.column_stack([np.ones(80), x])
beta = np.linalg.solve(X.T @ X, X.T @ y)
s2 = ((y - X @ beta) ** 2).sum() / 78
se_matrix = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
se_formula = np.sqrt(s2 / ((x - x.mean()) ** 2).sum())
print(round(se_matrix, 6), round(se_formula, 6))

0.039062 0.039062


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
a = rng.normal(10.00, 3, 200000); b = rng.normal(10.02, 3, 200000)
full = stats.ttest_ind(b, a, equal_var=False)
summ = stats.ttest_ind_from_stats(b.mean(), b.std(ddof=1), b.size, a.mean(), a.std(ddof=1), a.size, equal_var=False)
print(round(full.pvalue, 8), round(summ.pvalue, 8), np.isclose(full.statistic, summ.statistic))

0.03389195 0.03389195 True


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch18.html